<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase32_Controlled_Evidence_Loss_Drive_to_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 — Phase 32
## Controlled MRI Evidence-Loss Challenges and Source-Proxy Cohort Counterfactuals
**Google Drive → Google Drive · synthetic only**

### Entry gate: what Phase 31 actually established
- 10/10 synthetic **shadow structure** checks passed.
- 19 Athena binding decisions remained unreviewed; 36 original vocabulary gaps are still blocked.
- No completed Arm A, pyOMOP load, official MI-CDM load, real ADNI processing, or official target cohort-equivalence result.

**Independent experiment enabled by that gate:** freeze a source-evidence reference and inject controlled faults into a **target-like synthetic shadow export**. Compare a value-only eligibility proxy with a provenance- and contract-qualified source proxy. This tests the challenge protocol and its implementation; it does not substitute for the real four-arm experiment.

**Dataset:** retain both Phase 29 synthetic persons; add one **explicitly generated synthetic negative-control person** with an incomplete 12-month hippocampal pair. Their MCI status is **unknown**. Never describe the proxy as an eligible MCI cohort.

**Ground-truth limitation:** the test cases and detector are both investigator-designed, and the detector has access to a frozen clean source manifest. This is an **oracle-anchored debugging experiment**, not independent, blinded or clinical validation. Case results are deliberately not interpreted as population-level sensitivity, PPV or improved scientific inference.

**Safeguards:** no fabricated standard concepts, no protected Athena data, no real ADNI identifiers, no official target mapping claims. Arm C remains nonportable and unexecuted.

In [1]:
# Run in Colab. A local-only override exists for repeatable developer tests.
import os
DRIVE_ROOT_OVERRIDE = os.environ.get('ICHI2027_DRIVE_ROOT', '').strip()
if not DRIVE_ROOT_OVERRIDE:
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import json, hashlib, csv, copy, uuid
from collections import Counter, defaultdict
from datetime import datetime, timezone
import pandas as pd

DRIVE_ROOT = Path(DRIVE_ROOT_OVERRIDE) if DRIVE_ROOT_OVERRIDE else Path('/content/drive/MyDrive')
assert DRIVE_ROOT.is_dir(), f'Google Drive root not available: {DRIVE_ROOT}'

# Restrict scanning to the folder if it is known, to avoid stale Drive entries.
# Optional: set PROJECT_FOLDER_OVERRIDE to a folder within your MyDrive.
PROJECT_FOLDER_OVERRIDE = ''  # e.g. 'NeuroFHIR_ICHI2027/phase24_outputs'
SCAN_ROOT = DRIVE_ROOT / PROJECT_FOLDER_OVERRIDE if PROJECT_FOLDER_OVERRIDE else DRIVE_ROOT
assert SCAN_ROOT.is_dir(), 'Project folder override does not exist.'

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''): h.update(chunk)
    return h.hexdigest()

def valid_files(pattern):
    # pathlib.rglob on mounted Drive occasionally returns deleted/stale entries.
    try: candidates=list(SCAN_ROOT.rglob(pattern))
    except OSError: candidates=[]
    good=[]
    for p in candidates:
        try:
            if p.is_file() and p.stat().st_size>0: good.append(p)
        except (OSError, FileNotFoundError): continue
    return good

def read_json(path):
    with open(path,encoding='utf-8') as f: return json.load(f)

p31_choices=[]
for p in valid_files('ICHI2027_Phase31_SHAREABLE*.json'):
    try:
        data=read_json(p)
        if data.get('phase')=='31_frozen_athena_intake_source_semantics_and_synthetic_shadow_gate':
            p31_choices.append((data.get('run_utc',''),p,data))
    except (ValueError, OSError): continue
assert p31_choices, 'No valid Phase 31 SHAREABLE JSON found under selected Drive folder.'
_, P31_JSON, p31 = max(p31_choices,key=lambda item:item[0])
WORKDIR=P31_JSON.parent
P31_TESTS=WORKDIR/'ICHI2027_Phase31_SHADOW_STRUCTURE_TESTS.csv'
P31_BINDINGS=WORKDIR/'ICHI2027_Phase31_BINDING_REVIEW_STATUS.csv'
assert P31_TESTS.is_file() and P31_BINDINGS.is_file(), 'Phase 31 companion CSVs must be beside its shareable JSON.'

p29_matches=[]
for p in valid_files('ICHI2027_Phase29_SHAREABLE*.json'):
    try:
        if sha256(p)==p31['phase29_sha256']: p29_matches.append(p)
    except (OSError,KeyError): pass
assert p29_matches, 'Matching Phase 29 SHAREABLE JSON not found; cannot mix versions.'
P29_JSON=next((p for p in p29_matches if p.parent==WORKDIR), p29_matches[0])
p29=read_json(P29_JSON)
P29_BUNDLE=P29_JSON.parent/'ICHI2027_Phase29_SYNTHETIC_FHIR_WITH_PROCEDURES.json'
P29_SIDECAR=P29_JSON.parent/'ICHI2027_Phase29_SYNTHETIC_LINEAGE_SIDECAR.csv'
assert P29_BUNDLE.is_file() and P29_SIDECAR.is_file(), 'Missing matching Phase 29 FHIR bundle or sidecar in its source folder.'
assert sha256(P29_BUNDLE)==p29['synthetic_fixture']['patched_bundle_sha256'], 'Phase 29 bundle hash mismatch; refusing mixed source inputs.'
assert p29['pinned_imagewg_commit']==p31['pinned_imagewg_commit']
assert p29['pinned_omop55_commit']==p31['pinned_omop55_commit']

struct=pd.read_csv(P31_TESTS)
bindings=pd.read_csv(P31_BINDINGS)
sidecar=pd.read_csv(P29_SIDECAR,dtype={'source_observation_id':'string'})
assert len(struct)==10 and struct['passed'].astype(str).str.lower().eq('true').all(), 'Phase31 structural gate not passed.'
assert len(bindings)==p31['private_vocab_intake']['total_original_binding_keys']==19
assert sidecar['source_observation_id'].nunique()==16 and len(sidecar)==16
assert p31['gates']['synthetic_shadow_structural_tests_passed'] is True
assert p31['gates']['real_MRI_rows_processed']==0
assert p31['gates']['synthetic_faithful_armA_completed'] is False
fhir=read_json(P29_BUNDLE)
print('Phase 31 verified: 10/10 structural checks; 19 binding statuses read.')
print('Matched Phase 29 FHIR and lineage hashes. Saving Phase 32 beside Phase 31:')
print(WORKDIR)

Phase 31 verified: 10/10 structural checks; 19 binding statuses read.
Matched Phase 29 FHIR and lineage hashes. Saving Phase 32 beside Phase 31:
/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs


## Freeze a source manifest and add one synthetic negative control

The original Phase 29 FHIR bundle contains two synthetic people with both hippocampal hemispheres at baseline and 12 months. Phase 32 clones only synthetic resource content to create a **third, explicitly manufactured** person lacking the 12-month right hippocampal observation. All new patient IDs, resource references, DICOM-style identifiers and source rows are synthetic. Original Phase 29 observations are reconciled against the Phase 29 lineage sidecar before experimentation.

In [3]:
# Duplicate only the first *synthetic* person and deterministically relabel every reference.
# Omit one synthetic follow-up hippocampal observation and its Provenance,
# yielding a prespecified source-negative control without introducing ADNI data.
ORIG='syn-p01'; AUG='aug-syn-p03'
selected=[]
for item in fhir['entry']:
    r=item['resource']
    if ORIG in str(r.get('id','')):
        selected.append(copy.deepcopy(item))
assert selected and any(e['resource']['resourceType']=='Patient' for e in selected)

def replace_strings(obj):
    if isinstance(obj,str): return obj.replace(ORIG,AUG)
    if isinstance(obj,list): return [replace_strings(v) for v in obj]
    if isinstance(obj,dict): return {k:replace_strings(v) for k,v in obj.items()}
    return obj

aug_entries=[replace_strings(e) for e in selected]
new_bundle=copy.deepcopy(fhir)
new_bundle['identifier']={'system':'https://example.org/ichi2027/synthetic/bundle','value':'phase32-expanded-three-person'}

for e in aug_entries:
    r=e['resource']
    if r['resourceType']=='ImagingStudy':
        sid=r['id']
        uid='2.25.'+str(uuid.uuid5(uuid.NAMESPACE_URL,'ichi2027-phase32-study-'+sid).int)
        seriesuid='2.25.'+str(uuid.uuid5(uuid.NAMESPACE_URL,'ichi2027-phase32-series-'+sid).int)
        r['identifier']=[i for i in r.get('identifier',[]) if i.get('system')!='urn:dicom:uid']
        r['identifier'].append({'system':'urn:dicom:uid','value':'urn:oid:'+uid})
        assert len(r.get('series',[]))==1
        r['series'][0]['uid']=seriesuid

missing_obs_id='obs-'+AUG+'-hippocampus-right-m12'
aug_entries=[e for e in aug_entries if e['resource'].get('id') not in {missing_obs_id,'prov-'+missing_obs_id}]
new_bundle['entry'].extend(aug_entries)

idx={}
for e in new_bundle['entry']:
    r=e['resource']; key=r.get('resourceType','')+'/'+r.get('id','')
    assert key not in idx, f'Generated duplicate FHIR ID {key}'
    idx[key]=r

patients={k:v for k,v in idx.items() if k.startswith('Patient/')}
studies={k:v for k,v in idx.items() if k.startswith('ImagingStudy/')}
procedures={k:v for k,v in idx.items() if k.startswith('Procedure/')}
devices={k:v for k,v in idx.items() if k.startswith('Device/')}
observations={k:v for k,v in idx.items() if k.startswith('Observation/')}
assert (len(patients),len(studies),len(procedures),len(observations))==(3,6,6,23)

original_obs={e['resource']['id'] for e in fhir['entry'] if e['resource']['resourceType']=='Observation'}
assert set(sidecar['source_observation_id'])==original_obs

def extension(r,suffix,key):
    matches=[x.get(key) for x in r.get('extension',[]) if x.get('url','').endswith('/'+suffix)]
    return matches[0] if len(matches)==1 else None

def source_manifest_row(o):
    oid=o['id']; pref=o['subject']['reference']; sref=o['focus'][0]['reference']
    dref=o['device']['reference']
    assert pref in patients and sref in studies and dref in devices
    st=studies[sref]
    assert st['subject']['reference']==pref
    proc_ref=st['procedureReference']['reference']
    assert proc_ref in procedures and procedures[proc_ref]['subject']['reference']==pref
    assert len(st['series'])==1
    uid=[x['value'] for x in st['identifier'] if x['system']=='urn:dicom:uid']
    assert len(uid)==1 and uid[0].startswith('urn:oid:2.25.')
    provs=[r for k,r in idx.items() if k.startswith('Provenance/') and
           any(t.get('reference')=='Observation/'+oid for t in r.get('target',[]))]
    assert len(provs)==1
    ents={e.get('what',{}).get('reference') for e in provs[0].get('entity',[])}
    assert sref in ents and dref in ents
    anatomy=o['bodySite']['coding'][0]['code'].upper()
    if '-HIP-' in anatomy: region='HIP'
    elif '-ENT-' in anatomy: region='ENT'
    else: raise AssertionError('Unexpected synthetic anatomy')
    laterality='LEFT' if anatomy.endswith('-LEFT') else 'RIGHT' if anatomy.endswith('-RIGHT') else None
    assert laterality is not None
    visit=extension(o,'timepoint-label','valueString')
    assert visit in {'baseline','12-month'}
    val=o['valueQuantity']
    assert val.get('unit')=='mm3' and val.get('system')=='http://unitsofmeasure.org'
    return {
      'observation_id':oid,'patient_ref':pref,'study_ref':sref,'procedure_ref':proc_ref,
      'device_ref':dref,'region':region,'laterality':laterality,'timepoint':visit,
      'volume_value':float(val['value']),'unit':'mm3',
      'model_version':devices[dref]['version'][0]['value'],
      'trace_state':extension(o,'trace-state','valueCode'),
      'trace_rule':extension(o,'trace-rule-version','valueString'),
      'study_uid':uid[0], 'series_uid':st['series'][0]['uid'],
    }

source_rows=[source_manifest_row(o) for o in observations.values()]
source_rows=sorted(source_rows,key=lambda r:r['observation_id'])
source_manifest={r['observation_id']:r for r in source_rows}
assert len(source_manifest)==23
assert sum(r['patient_ref']=='Patient/'+AUG for r in source_rows)==7
for r in source_rows:
    if r['observation_id'] in original_obs:
        old=sidecar.loc[sidecar['source_observation_id']==r['observation_id']].iloc[0]
        assert r['study_ref']=='ImagingStudy/'+old['source_study_id']
        assert r['procedure_ref']=='Procedure/'+old['synthetic_fhir_procedure_id']
        assert r['region']==old['anatomy'] and r['laterality']==old['laterality']
        assert r['timepoint']==old['baseline_or_followup']
        assert abs(r['volume_value']-float(old['value']))<1e-9
        assert r['model_version']==str(old['algorithm_version'])

# The cohort used here is a *source-only bilateral hippocampal evidence proxy*,
# never the blueprint's clinical MCI eligibility cohort.
NEEDED={('HIP',side,visit) for side in ('LEFT','RIGHT') for visit in ('baseline','12-month')}

def membership_from_rows(rows):
    grouped=defaultdict(list)
    for row in rows:
        if row['region']=='HIP': grouped[row['patient_ref']].append(row)
    eligible=set()
    for pid,items in grouped.items():
        observed=Counter((r['region'],r['laterality'],r['timepoint']) for r in items)
        if all(observed[x]>=1 for x in NEEDED): eligible.add(pid)
    return eligible

REF_MEMBERS=membership_from_rows(source_rows)
assert REF_MEMBERS=={'Patient/syn-p01','Patient/syn-p02'}
assert 'Patient/'+AUG not in REF_MEMBERS
print('Source fixture: 3 fully synthetic people, 6 imaging studies, 23 observations.')
print('Source-only proxy membership: 2 positive and 1 intentionally incomplete negative control; MCI unknown.')

Source fixture: 3 fully synthetic people, 6 imaging studies, 23 observations.
Source-only proxy membership: 2 positive and 1 intentionally incomplete negative control; MCI unknown.


## Prespecify synthetic corruption and controls

The cases below are **registered before the detector runs**. `E1` deliberately swaps left/right labels while leaving each label structurally valid: without a frozen source manifest, a simple presence-only checker could miss this error. `E2_valid_conversion` is a legitimate normalization control, while the entorhinal-only negative control tests whether a hippocampal-specific policy unnecessarily excludes a person for an unrelated observation fault. The extra-record case tests fabricated target-side evidence and false inclusion.

In [4]:
# Fully deterministic case construction; all edits are to an in-memory synthetic shadow.
def clone(): return copy.deepcopy(source_rows)
def get(rows,pid,region,side,visit):
    found=[r for r in rows if r['patient_ref']=='Patient/'+pid and r['region']==region and r['laterality']==side and r['timepoint']==visit]
    assert len(found)==1,(pid,region,side,visit,len(found))
    return found[0]

def make_cases():
    cases=[]
    def add(name,rows,expected,strict_members,description):
        cases.append({'case':name,'rows':rows,'expected':set(expected),
                      'strict_members':set('Patient/'+p for p in strict_members),'description':description})
    good=['syn-p01','syn-p02']
    add('E0_clean',clone(),[],good,'Untouched source-derived synthetic shadow')
    b=clone(); l=get(b,'syn-p01','HIP','LEFT','baseline');r=get(b,'syn-p01','HIP','RIGHT','baseline')
    l['laterality'],r['laterality']=r['laterality'],l['laterality']
    add('E1_bilateral_label_swap',b,['ANATOMY_LATERALITY_MISMATCH'],['syn-p02'],'Two valid but swapped baseline hemisphere labels')
    b=clone();r=get(b,'syn-p01','HIP','LEFT','baseline');r['volume_value']/=1000;r['unit']='cm3'
    add('E2_valid_unit_normalization',b,[],good,'Correct mm3-to-cm3 conversion; no consequential loss')
    b=clone();get(b,'syn-p01','HIP','RIGHT','baseline')['unit']=None
    add('E2_missing_unit',b,['UNIT_MISSING'],['syn-p02'],'Baseline right hippocampal volume loses its unit')
    b=clone();get(b,'syn-p01','HIP','RIGHT','12-month')['study_ref']='ImagingStudy/study-syn-p02-m12'
    add('E3_cross_person_study_link',b,['STUDY_LINK_MISMATCH','CROSS_PERSON_STUDY_LINK'],['syn-p02'],'Right hippocampal follow-up linked to another synthetic patient')
    b=clone();get(b,'syn-p01','HIP','LEFT','baseline')['model_version']=None
    add('E4_missing_model_version',b,['MODEL_VERSION_LOSS'],['syn-p02'],'Extraction algorithm version removed')
    b=clone();get(b,'syn-p01','HIP','RIGHT','baseline')['trace_state']=None
    add('E5_reliability_state_removed',b,['TRACE_STATE_LOSS'],['syn-p02'],'Original recorded TRACE state removed')
    b=clone();b.append(copy.deepcopy(get(b,'syn-p01','HIP','LEFT','baseline')))
    add('E6_duplicate_observation',b,['DUPLICATE_OBSERVATION_ID'],['syn-p02'],'Duplicate target-side observation identifier')
    b=clone();get(b,'syn-p01','HIP','LEFT','baseline')['study_ref']='ImagingStudy/does-not-exist'
    add('E6_unresolved_study_link',b,['STUDY_LINK_MISMATCH','SOURCE_LINK_UNRESOLVED'],['syn-p02'],'Unresolved synthetic imaging-source reference')
    b=clone();r=get(b,'syn-p02','HIP','LEFT','baseline');r['laterality']='RIGHT';r['model_version']=None;r['trace_state']=None
    add('E7_combined_fault',b,['ANATOMY_LATERALITY_MISMATCH','MODEL_VERSION_LOSS','TRACE_STATE_LOSS'],['syn-p01'],'Combined synthetic laterality/provenance/reliability corruption')
    b=clone();get(b,'syn-p01','ENT','LEFT','baseline')['laterality']='RIGHT'
    add('NC_entorhinal_noncohort_fault',b,['ANATOMY_LATERALITY_MISMATCH'],good,'Unrelated entorhinal corruption should not remove hippocampal-only eligibility')
    b=clone()
    extra=copy.deepcopy(get(b,'syn-p02','HIP','RIGHT','12-month'))
    extra['observation_id']='shadow-injected-unobserved-hip-right-m12'
    extra['patient_ref']='Patient/'+AUG
    extra['study_ref']='ImagingStudy/study-'+AUG+'-m12'
    extra['procedure_ref']='Procedure/proc-study-'+AUG+'-m12'
    extra['device_ref']='Device/dev-'+AUG
    augstudy=studies[extra['study_ref']]
    extra['study_uid']=next(x['value'] for x in augstudy['identifier'] if x['system']=='urn:dicom:uid')
    extra['series_uid']=augstudy['series'][0]['uid']
    b.append(extra)
    add('NC_injected_false_inclusion',b,['UNEXPECTED_TARGET_FEATURE'],good,'Target-only extra row makes an incomplete person appear eligible to a value-only query')
    return cases

cases=make_cases()
assert len(cases)==12
# Keep case definitions distinct from the detector's implementation.
ground_truth=[{'case':c['case'],'fault_description':c['description'],
               'expected_consequential_warnings':';'.join(sorted(c['expected'])),
               'expected_strict_positive_person_count':len(c['strict_members'])}
              for c in cases]
ground_truth_json=json.dumps(ground_truth,sort_keys=True,separators=(',',':'))
GROUND_TRUTH_SHA256=hashlib.sha256(ground_truth_json.encode()).hexdigest()
print('12 controlled cases frozen. Ground-truth SHA256:',GROUND_TRUTH_SHA256)

12 controlled cases frozen. Ground-truth SHA256: f68589a971ac8f1ea84846c6c58b6e5eeb06a93574a942c8e41ee416d1f4b8b0


## Oracle-anchored detector and policy tests

The detector compares the target-like shadow to the **frozen clean FHIR-derived manifest**. It detects mismatches that a value-only query would overlook. Strict source-proxy membership requires clean evidence for the four hippocampal slots; entorhinal-only problems do not exclude a person from the hippocampal proxy. Do not present strict policy exclusion caused by injected corruption as evidence of a true clinical exclusion.

In [5]:
# This is a source-manifest audit, NOT the full future Arm D nor a trained or blinded detector.
ALLOWED_UNITS={'mm3':1.0,'cm3':1000.0}
SEVERE={
 'DUPLICATE_OBSERVATION_ID','SOURCE_OBSERVATION_MISSING','UNEXPECTED_TARGET_FEATURE',
 'PATIENT_MISMATCH','STUDY_LINK_MISMATCH','CROSS_PERSON_STUDY_LINK',
 'SOURCE_LINK_UNRESOLVED','PROCEDURE_LINK_MISMATCH',
 'ANATOMY_LATERALITY_MISMATCH','REGION_MISMATCH','TIMEPOINT_MISMATCH',
 'UNIT_MISSING','UNIT_UNSUPPORTED','VALUE_MISSING','VALUE_UNIT_DISAGREEMENT',
 'MODEL_VERSION_LOSS','MODEL_IDENTITY_MISMATCH','TRACE_STATE_LOSS',
 'TRACE_STATE_MISMATCH','TRACE_RULE_LOSS','TRACE_RULE_MISMATCH',
 'STUDY_UID_MISMATCH','SERIES_UID_MISMATCH',
}

def audit_shadow(rows):
    result=[]; normalized=0
    ct=Counter(x['observation_id'] for x in rows)
    def emit(rid,pid,code):
        assert code in SEVERE
        result.append({'observation_id':rid,'patient_ref':pid,'warning_code':code,'severity':'CONSEQUENTIAL_SYNTHETIC'})
    for oid,n in ct.items():
        if n>1:
            sample=next(x for x in rows if x['observation_id']==oid)
            emit(oid,sample['patient_ref'],'DUPLICATE_OBSERVATION_ID')
    for oid,src in source_manifest.items():
        if ct[oid]==0: emit(oid,src['patient_ref'],'SOURCE_OBSERVATION_MISSING')
    for r in rows:
        oid=r['observation_id']
        if oid not in source_manifest:
            emit(oid,r['patient_ref'],'UNEXPECTED_TARGET_FEATURE')
            continue
        src=source_manifest[oid]
        if r['patient_ref']!=src['patient_ref']: emit(oid,src['patient_ref'],'PATIENT_MISMATCH')
        if r['study_ref']!=src['study_ref']: emit(oid,src['patient_ref'],'STUDY_LINK_MISMATCH')
        if r['study_ref'] not in studies: emit(oid,src['patient_ref'],'SOURCE_LINK_UNRESOLVED')
        elif studies[r['study_ref']]['subject']['reference']!=src['patient_ref']:
            emit(oid,src['patient_ref'],'CROSS_PERSON_STUDY_LINK')
        if r['procedure_ref']!=src['procedure_ref']: emit(oid,src['patient_ref'],'PROCEDURE_LINK_MISMATCH')
        if r['region']!=src['region']: emit(oid,src['patient_ref'],'REGION_MISMATCH')
        if r['laterality']!=src['laterality']:
            emit(oid,src['patient_ref'],'ANATOMY_LATERALITY_MISMATCH')
        if r['timepoint']!=src['timepoint']: emit(oid,src['patient_ref'],'TIMEPOINT_MISMATCH')
        if r['unit'] is None or str(r['unit']).strip()=='':
            emit(oid,src['patient_ref'],'UNIT_MISSING')
        elif r['unit'] not in ALLOWED_UNITS:
            emit(oid,src['patient_ref'],'UNIT_UNSUPPORTED')
        elif r.get('volume_value') is None:
            emit(oid,src['patient_ref'],'VALUE_MISSING')
        else:
            normalized_value=float(r['volume_value'])*ALLOWED_UNITS[r['unit']]
            if abs(normalized_value-src['volume_value'])>max(1e-6,src['volume_value']*1e-9):
                emit(oid,src['patient_ref'],'VALUE_UNIT_DISAGREEMENT')
            elif r['unit']!=src['unit']: normalized+=1
        if not r.get('model_version'):
            emit(oid,src['patient_ref'],'MODEL_VERSION_LOSS')
        elif r['model_version']!=src['model_version']:
            emit(oid,src['patient_ref'],'MODEL_IDENTITY_MISMATCH')
        if not r.get('trace_state'):
            emit(oid,src['patient_ref'],'TRACE_STATE_LOSS')
        elif r['trace_state']!=src['trace_state']:
            emit(oid,src['patient_ref'],'TRACE_STATE_MISMATCH')
        if not r.get('trace_rule'): emit(oid,src['patient_ref'],'TRACE_RULE_LOSS')
        elif r['trace_rule']!=src['trace_rule']:
            emit(oid,src['patient_ref'],'TRACE_RULE_MISMATCH')
        if r['study_uid']!=src['study_uid']: emit(oid,src['patient_ref'],'STUDY_UID_MISMATCH')
        if r['series_uid']!=src['series_uid']: emit(oid,src['patient_ref'],'SERIES_UID_MISMATCH')
    return result,normalized

def strict_membership(rows,warnings):
    affected={(w['patient_ref'],w['observation_id']) for w in warnings}
    out=set()
    for pid in patients:
        # Reject target-only records that would create a false inclusion.
        person_rows=[r for r in rows if r['patient_ref']==pid and r['region']=='HIP']
        slots=defaultdict(list)
        for r in person_rows:
            slots[(r['region'],r['laterality'],r['timepoint'])].append(r)
        if any(len(slots[s])!=1 for s in NEEDED): continue
        required=[slots[s][0] for s in NEEDED]
        if any(r['observation_id'] not in source_manifest for r in required): continue
        if any(source_manifest[r['observation_id']]['patient_ref']!=pid for r in required): continue
        if any((pid,r['observation_id']) in affected for r in required): continue
        out.add(pid)
    return out

def member_metrics(candidate,reference):
    union=candidate|reference
    return {
       'n':len(candidate), 'false_inclusions':len(candidate-reference),
       'false_exclusions':len(reference-candidate),
       'jaccard':len(candidate&reference)/len(union) if union else 1.0
    }

case_results=[];event_results=[];cohort_results=[]
for c in cases:
    found,normalized=audit_shadow(c['rows'])
    observed={w['warning_code'] for w in found}
    value_only=membership_from_rows(c['rows'])
    contract=strict_membership(c['rows'],found)
    assert c['expected'].issubset(observed), (c['case'],c['expected']-observed)
    assert contract==c['strict_members'], (c['case'],'unexpected strict membership',contract,c['strict_members'])
    if c['case'] in {'E0_clean','E2_valid_unit_normalization'}:
        assert len(found)==0, (c['case'],found)
    if c['case']=='E2_valid_unit_normalization': assert normalized==1
    if c['case']=='NC_injected_false_inclusion':
        assert 'Patient/'+AUG in value_only and 'Patient/'+AUG not in contract
    if c['case']=='NC_entorhinal_noncohort_fault': assert contract==REF_MEMBERS
    vm=member_metrics(value_only,REF_MEMBERS); cm=member_metrics(contract,REF_MEMBERS)
    case_results.append({
      'case':c['case'],'fixture_passed':True,'reference_proxy_n':len(REF_MEMBERS),
      'shadow_row_count':len(c['rows']),'expected_fault_code_count':len(c['expected']),
      'expected_fault_codes_detected':len(c['expected']&observed),
      'detected_warning_event_count':len(found),'detected_warning_types':';'.join(sorted(observed)),
      'legitimate_unit_normalizations':normalized,
      'value_only_proxy_n':vm['n'],'value_only_false_inclusions':vm['false_inclusions'],
      'value_only_false_exclusions':vm['false_exclusions'],'value_only_jaccard':vm['jaccard'],
      'contract_proxy_n':cm['n'],'contract_false_inclusions':cm['false_inclusions'],
      'contract_false_exclusions':cm['false_exclusions'],'contract_jaccard':cm['jaccard'],
      'is_oracle_anchored_debug_test':True,'official_omop_cohort_equivalence':False,
    })
    for item in found: event_results.append({'case':c['case'],**item})
    for pid in sorted(patients):
        cohort_results.append({'case':c['case'],'synthetic_patient_id':pid,
           'source_proxy_member':pid in REF_MEMBERS,
           'value_only_shadow_member':pid in value_only,
           'contract_qualified_shadow_member':pid in contract,
           'MCI_status':'UNKNOWN_NOT_ATTESTED',
        })

case_df=pd.DataFrame(case_results)
event_df=pd.DataFrame(event_results,columns=['case','observation_id','patient_ref','warning_code','severity'])
cohort_df=pd.DataFrame(cohort_results)
assert case_df['fixture_passed'].all() and len(cohort_df)==12*3
assert not cohort_df['MCI_status'].ne('UNKNOWN_NOT_ATTESTED').any()
print('PASS: 12/12 controlled synthetic challenge cases matched prespecified warnings and policy behavior.')
print('Source-only proxy membership:',len(REF_MEMBERS),'of',len(patients),'fully synthetic people.')
display(case_df[['case','detected_warning_types','value_only_proxy_n','contract_proxy_n','value_only_false_inclusions']])

PASS: 12/12 controlled synthetic challenge cases matched prespecified warnings and policy behavior.
Source-only proxy membership: 2 of 3 fully synthetic people.


,case,detected_warning_types,value_only_proxy_n,contract_proxy_n,value_only_false_inclusions
0,E0_clean,,2,2,0
1,E1_bilateral_label_swap,ANATOMY_LATERALITY_MISMATCH,2,1,0
2,E2_valid_unit_normalization,,2,2,0
3,E2_missing_unit,UNIT_MISSING,2,1,0
4,E3_cross_person_study_link,CROSS_PERSON_STUDY_LINK;STUDY_LINK_MISMATCH,2,1,0
5,E4_missing_model_version,MODEL_VERSION_LOSS,2,1,0
6,E5_reliability_state_removed,TRACE_STATE_LOSS,2,1,0
7,E6_duplicate_observation,DUPLICATE_OBSERVATION_ID,2,1,0
8,E6_unresolved_study_link,SOURCE_LINK_UNRESOLVED;STUDY_LINK_MISMATCH,2,1,0
9,E7_combined_fault,ANATOMY_LATERALITY_MISMATCH;MODEL_VERSION_LOSS...,1,1,0


## Save public-safe Drive outputs with full auditability

Outputs intentionally contain only synthetic identifiers and aggregate test details. The Phase 31 Athena gate remains unchanged. `SOURCE_PROXY_COHORT.csv` describes an intentionally simplified evidence proxy, **not** the blueprint's clinical MCI cohort and **not** OMOP target cohort equivalence.

In [6]:
OUTPUTS={
 'shareable':WORKDIR/'ICHI2027_Phase32_SHAREABLE.json',
 'ground_truth':WORKDIR/'ICHI2027_Phase32_GROUND_TRUTH.csv',
 'cases':WORKDIR/'ICHI2027_Phase32_CASE_RESULTS.csv',
 'events':WORKDIR/'ICHI2027_Phase32_EVENT_RESULTS.csv',
 'cohort':WORKDIR/'ICHI2027_Phase32_SOURCE_PROXY_COHORT.csv',
 'manifest':WORKDIR/'ICHI2027_Phase32_EXPANDED_SOURCE_MANIFEST.csv',
 'bundle':WORKDIR/'ICHI2027_Phase32_EXPANDED_SYNTHETIC_FHIR_BUNDLE.json',
}

def json_bytes(o): return json.dumps(o,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode('utf-8')

with open(OUTPUTS['bundle'],'w',encoding='utf-8') as f: json.dump(new_bundle,f,indent=2)
pd.DataFrame(source_rows).to_csv(OUTPUTS['manifest'],index=False)
pd.DataFrame(ground_truth).to_csv(OUTPUTS['ground_truth'],index=False)
case_df.to_csv(OUTPUTS['cases'],index=False)
event_df.to_csv(OUTPUTS['events'],index=False)
cohort_df.to_csv(OUTPUTS['cohort'],index=False)

summary={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'32_source_manifest_controlled_synthetic_evidence_loss_challenges',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'privacy':'Public synthetic-only: no ADNI, personal identifiers, private Athena files or private Drive paths.',
 'upstream':{
   'phase31_shareable_sha256':sha256(P31_JSON),
   'phase29_shareable_sha256':sha256(P29_JSON),
   'phase29_synthetic_bundle_sha256':sha256(P29_BUNDLE),
   'phase31_shadow_checks_verified':10,
   'imagewg_pinned_commit':p31['pinned_imagewg_commit'],
   'omop55_pinned_commit':p31['pinned_omop55_commit'],
 },
 'synthetic_fixture':{
   'original_synthetic_persons':2,'explicitly_augmented_negative_control_persons':1,
   'total_synthetic_persons':3,'source_imaging_studies':len(studies),
   'source_observations':len(source_rows),'source_proxy_eligible_persons':len(REF_MEMBERS),
   'source_MCI_status_attested':False,
   'manifest_sha256':hashlib.sha256(json_bytes(source_rows)).hexdigest(),
   'expanded_bundle_sha256':sha256(OUTPUTS['bundle']),
   'ground_truth_sha256':GROUND_TRUTH_SHA256,
 },
 'experiments':{
   'prespecified_case_count':len(cases),'cases_passing_declared_expectations':int(case_df['fixture_passed'].sum()),
   'method':'ORACLE_ANCHORED_SYNTHETIC_SHADOW_DEBUG_TEST',
   'blinded':False, 'independent_heldout_validation':False,
   'unit_normalization_control_passed':bool(case_df.loc[case_df['case']=='E2_valid_unit_normalization','legitimate_unit_normalizations'].iloc[0]==1),
   'negative_control_region_specific_policy_passed':True,
   'fabricated_target_feature_false_inclusion_detected':True,
   'clinical_detection_performance_estimated':False,
 },
 'athena_gate':{
   'frozen_private_snapshot_provided_in_phase31':bool(p31['private_vocab_intake']['frozen_CONCEPT_supplied']),
   'original_bindings_total':int(p31['private_vocab_intake']['total_original_binding_keys']),
   'reviewed_snapshot_checked_keys_in_phase31':int(p31['private_vocab_intake']['snapshot_and_source_checked_keys']),
   'original_vocab_gaps_remaining_blocked':int(p31['original_48_gap_disposition']['vocabulary_still_blocked']),
 },
 'gates':{
   'phase31_verified':True, 'synthetic_manifest_original_16_reconciled':True,
   'generated_third_person_explicitly_synthetic':True,'controlled_shadow_challenge_cases_passed':True,
   'official_target_cohort_equivalence_tested':False,
   'synthetic_faithful_armA_mapping_completed':False,
   'original_generic_armC_executed':False,
   'pyOMOP_ETL_executed':False,'official_MICDM_load_completed':False,
   'real_MRI_rows_processed':0,'true_four_arm_comparison_completed':False,
 },
 'public_output_sha256':{k:sha256(v) for k,v in OUTPUTS.items() if k!='shareable'},
 'next_stage_gate':(
   'Use a private authorized frozen CONCEPT.csv and independently reviewed source-supported bindings to unlock '
   'a faithful synthetic Arm A; until then keep official OMOP/MI-CDM loading and cohort equivalence blocked. '
   'The Phase32 source-proxy challenge cases may be reused as prespecified test fixtures, '
   'but are not an independent blinded performance evaluation.'
 )
}
with open(OUTPUTS['shareable'],'w',encoding='utf-8') as f: json.dump(summary,f,indent=2)

# Drive readback, input integrity and privacy/claim boundaries.
again=read_json(OUTPUTS['shareable'])
assert again['synthetic_fixture']['expanded_bundle_sha256']==sha256(OUTPUTS['bundle'])
assert again['upstream']['phase31_shareable_sha256']==sha256(P31_JSON)
assert again['experiments']['prespecified_case_count']==12
assert again['gates']['official_target_cohort_equivalence_tested'] is False
assert again['gates']['true_four_arm_comparison_completed'] is False
assert again['gates']['real_MRI_rows_processed']==0
assert pd.read_csv(OUTPUTS['cases'])['fixture_passed'].astype(str).str.lower().eq('true').all()
assert len(pd.read_csv(OUTPUTS['ground_truth']))==12
assert len(pd.read_csv(OUTPUTS['cohort']))==36
assert sha256(OUTPUTS['bundle'])==again['public_output_sha256']['bundle']
print('PHASE 32 COMPLETE. Verified Drive readback of 7 public-safe files:')
for k,v in OUTPUTS.items(): print(' -',v.name)
print('Official OMOP/MI-CDM and real-data gates remain BLOCKED until independently verified.')

PHASE 32 COMPLETE. Verified Drive readback of 7 public-safe files:
 - ICHI2027_Phase32_SHAREABLE.json
 - ICHI2027_Phase32_GROUND_TRUTH.csv
 - ICHI2027_Phase32_CASE_RESULTS.csv
 - ICHI2027_Phase32_EVENT_RESULTS.csv
 - ICHI2027_Phase32_SOURCE_PROXY_COHORT.csv
 - ICHI2027_Phase32_EXPANDED_SOURCE_MANIFEST.csv
 - ICHI2027_Phase32_EXPANDED_SYNTHETIC_FHIR_BUNDLE.json
Official OMOP/MI-CDM and real-data gates remain BLOCKED until independently verified.
